# Module 4c: Data Handoff Patterns

**DSC 232R - Big Data Analysis Using Spark**

This notebook covers data transfer strategies:
1. Parquet file exchange
2. In-memory transfer via Pandas
3. Arrow-based transfer
4. Choosing the right strategy

## Key Takeaways

- **Parquet files** are the most robust handoff method
- **In-memory transfer** works for small to medium data
- **Arrow format** enables efficient columnar transfer
- Choose based on data size, latency needs, and infrastructure

In [ ]:
!pip install ray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 9.1 MB/s eta 0:00:00


In [ ]:
import ray
import numpy as np
import pandas as pd
import time
import os
import tempfile
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize frameworks
if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, logging_level="WARNING")

spark = SparkSession.builder \
    .appName("DataHandoff") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Create temp directory for files
TEMP_DIR = tempfile.mkdtemp(prefix="handoff_")
print(f"Temp directory: {TEMP_DIR}")

/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Temp directory: /tmp/handoff_th87rgzc


In [ ]:
# Generate sample data at different sizes
def generate_data(n_rows):
    np.random.seed(42)
    return pd.DataFrame({
        "id": range(n_rows),
        "feature1": np.random.randn(n_rows),
        "feature2": np.random.randn(n_rows),
        "feature3": np.random.randn(n_rows),
        "category": np.random.choice(["A", "B", "C", "D"], n_rows),
        "value": np.random.uniform(0, 100, n_rows),
    })

# Create datasets of different sizes
sizes = {
    "small": 10_000,
    "medium": 100_000,
    "large": 1_000_000,
}

datasets = {name: generate_data(n) for name, n in sizes.items()}

for name, df in datasets.items():
    mem_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"{name}: {len(df):,} rows, {mem_mb:.1f} MB")

small: 10,000 rows, 0.9 MB
medium: 100,000 rows, 8.6 MB
large: 1,000,000 rows, 85.8 MB


---

## 1. Pattern 1: Parquet File Exchange

The most robust and commonly used pattern.

In [ ]:
def handoff_via_parquet(pandas_df, output_path):
    """Spark writes parquet, Ray reads it."""

    # Spark: Create DataFrame and process
    spark_df = spark.createDataFrame(pandas_df)

    # Spark: Add some transformations
    processed = spark_df \
        .withColumn("combined", F.col("feature1") + F.col("feature2")) \
        .filter(F.col("value") > 10)

    # Spark: Write to parquet
    start = time.time()
    processed.write.mode("overwrite").parquet(output_path)
    write_time = time.time() - start

    # Ray: Read parquet
    start = time.time()
    ray_ds = ray.data.read_parquet(output_path)
    _ = ray_ds.count()  # Force evaluation
    read_time = time.time() - start

    return {
        "write_time": write_time,
        "read_time": read_time,
        "total_time": write_time + read_time,
        "row_count": ray_ds.count()
    }

# Test with medium dataset
parquet_path = os.path.join(TEMP_DIR, "parquet_test")
result = handoff_via_parquet(datasets["medium"], parquet_path)

print("Parquet Handoff Results:")
print(f"  Spark write: {result['write_time']:.3f}s")
print(f"  Ray read: {result['read_time']:.3f}s")
print(f"  Total: {result['total_time']:.3f}s")
print(f"  Rows transferred: {result['row_count']:,}")

Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-05-19 17:46:28,414	INFO parquet_datasource.py:1133 -- Estimated parquet encoding ratio is 1.216.
2026-05-19 17:46:28,416	INFO parquet_datasource.py:1193 -- Estimated parquet reader batch size at 2485514 rows
2026-05-19 17:46:29,387	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-05-19 17:46:29,418	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-05-19_17-45-25_832868_4047/logs/ray-data
2026-05-19 17:46:29,421	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AggregateNumRows[AggregateNumRows]
2026-05-19 17:46:29,443	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_sto

Parquet Handoff Results:
  Spark write: 10.614s
  Ray read: 17.922s
  Total: 28.536s
  Rows transferred: 90,135


### Parquet Advantages

- **Compression**: Smaller files on disk
- **Schema preservation**: Column types maintained
- **Partitioning**: Can partition by columns
- **Predicate pushdown**: Ray can filter while reading

In [ ]:
# Demonstrate partitioned parquet
def partitioned_handoff(pandas_df, output_path):
    """Write partitioned parquet for efficient querying."""

    spark_df = spark.createDataFrame(pandas_df)

    # Write partitioned by category
    spark_df.write \
        .mode("overwrite") \
        .partitionBy("category") \
        .parquet(output_path)

    print(f"Partitions created: {os.listdir(output_path)}")

    # Ray can read specific partitions
    ds_all = ray.data.read_parquet(output_path)
    ds_category_a = ray.data.read_parquet(f"{output_path}/category=A/")

    return {
        "all_rows": ds_all.count(),
        "category_a_rows": ds_category_a.count()
    }

partitioned_path = os.path.join(TEMP_DIR, "partitioned_test")
part_result = partitioned_handoff(datasets["medium"], partitioned_path)
print(f"\nAll rows: {part_result['all_rows']:,}")
print(f"Category A rows: {part_result['category_a_rows']:,}")

Partitions created: ['category=A', '_SUCCESS', 'category=C', '._SUCCESS.crc', 'category=D', 'category=B']


Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-05-19 17:46:50,975	INFO parquet_datasource.py:1133 -- Estimated parquet encoding ratio is 1.129.
2026-05-19 17:46:50,979	INFO parquet_datasource.py:1193 -- Estimated parquet reader batch size at 3273604 rows


Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-05-19 17:46:51,073	INFO parquet_datasource.py:1133 -- Estimated parquet encoding ratio is 1.129.
2026-05-19 17:46:51,076	INFO parquet_datasource.py:1193 -- Estimated parquet reader batch size at 3273604 rows
2026-05-19 17:46:51,094	INFO logging.py:416 -- Registered dataset logger for dataset dataset_5_0
2026-05-19 17:46:51,102	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_5_0. Full logs are in /tmp/ray/session_2026-05-19_17-45-25_832868_4047/logs/ray-data
2026-05-19 17:46:51,103	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_5_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AggregateNumRows[AggregateNumRows]
2026-05-19 17:46:51,163	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_5_0 =======
2026-05-19 17:46:51,165	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:46:51,168	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:46:5


All rows: 100,000
Category A rows: 25,056


---

## 2. Pattern 2: In-Memory via Pandas

Good for small to medium datasets when speed matters.

In [ ]:
def handoff_via_pandas(pandas_df):
    """Spark collects to pandas, Ray reads from pandas."""

    # Spark: Create and process DataFrame
    spark_df = spark.createDataFrame(pandas_df)
    processed = spark_df \
        .withColumn("combined", F.col("feature1") + F.col("feature2")) \
        .filter(F.col("value") > 10)

    # Spark: Collect to Pandas (CAUTION: must fit in driver memory!)
    start = time.time()
    pandas_result = processed.toPandas()
    collect_time = time.time() - start

    # Ray: Create dataset from Pandas
    start = time.time()
    ray_ds = ray.data.from_pandas(pandas_result)
    _ = ray_ds.count()  # Force evaluation
    create_time = time.time() - start

    return {
        "collect_time": collect_time,
        "create_time": create_time,
        "total_time": collect_time + create_time,
        "row_count": ray_ds.count()
    }

# Test with small dataset
result = handoff_via_pandas(datasets["small"])

print("Pandas Handoff Results (small):")
print(f"  Spark collect: {result['collect_time']:.3f}s")
print(f"  Ray create: {result['create_time']:.3f}s")
print(f"  Total: {result['total_time']:.3f}s")
print(f"  Rows transferred: {result['row_count']:,}")

Pandas Handoff Results (small):
  Spark collect: 1.420s
  Ray create: 0.033s
  Total: 1.454s
  Rows transferred: 9,015


In [ ]:
# Warning: toPandas() with large data
print("CAUTION: toPandas() brings ALL data to driver memory!")
print("")
print("Safe: < 10 GB (depends on driver memory)")
print("Risky: 10-50 GB")
print("Dangerous: > 50 GB (likely OOM)")
print("")
print("For large data, use Parquet handoff instead.")

CAUTION: toPandas() brings ALL data to driver memory!

Safe: < 10 GB (depends on driver memory)
Risky: 10-50 GB
Dangerous: > 50 GB (likely OOM)

For large data, use Parquet handoff instead.


---

## 3. Pattern 3: Arrow-Based Transfer

Efficient columnar format for intermediate transfer.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

def handoff_via_arrow(pandas_df, arrow_path):
    """Use Arrow format for efficient transfer."""

    # Spark: Create and process DataFrame
    spark_df = spark.createDataFrame(pandas_df)
    processed = spark_df \
        .withColumn("combined", F.col("feature1") + F.col("feature2")) \
        .filter(F.col("value") > 10)

    # Spark: Write as parquet (Arrow-compatible)
    start = time.time()
    processed.write.mode("overwrite").parquet(arrow_path)
    write_time = time.time() - start

    # Read with PyArrow directly (for demonstration)
    start = time.time()
    arrow_table = pq.read_table(arrow_path)
    arrow_time = time.time() - start

    # Convert to Ray Dataset
    start = time.time()
    ray_ds = ray.data.from_arrow(arrow_table)
    _ = ray_ds.count()
    ray_time = time.time() - start

    return {
        "write_time": write_time,
        "arrow_read_time": arrow_time,
        "ray_create_time": ray_time,
        "total_time": write_time + arrow_time + ray_time,
        "row_count": ray_ds.count()
    }

arrow_path = os.path.join(TEMP_DIR, "arrow_test")
result = handoff_via_arrow(datasets["medium"], arrow_path)

print("Arrow Handoff Results:")
print(f"  Spark write: {result['write_time']:.3f}s")
print(f"  Arrow read: {result['arrow_read_time']:.3f}s")
print(f"  Ray create: {result['ray_create_time']:.3f}s")
print(f"  Total: {result['total_time']:.3f}s")
print(f"  Rows transferred: {result['row_count']:,}")

Arrow Handoff Results:
  Spark write: 1.916s
  Arrow read: 0.045s
  Ray create: 0.045s
  Total: 2.006s
  Rows transferred: 90,135


---

## 4. Benchmark: All Patterns

In [ ]:
def benchmark_all_patterns(dataset_name, pandas_df):
    """Benchmark all handoff patterns."""
    results = {}

    # Pattern 1: Parquet
    path = os.path.join(TEMP_DIR, f"bench_{dataset_name}_parquet")
    parquet_result = handoff_via_parquet(pandas_df, path)
    results["Parquet"] = parquet_result["total_time"]

    # Pattern 2: Pandas (only for small/medium)
    if len(pandas_df) <= 500_000:
        pandas_result = handoff_via_pandas(pandas_df)
        results["Pandas"] = pandas_result["total_time"]
    else:
        results["Pandas"] = float('inf')  # Too large

    # Pattern 3: Arrow
    path = os.path.join(TEMP_DIR, f"bench_{dataset_name}_arrow")
    arrow_result = handoff_via_arrow(pandas_df, path)
    results["Arrow"] = arrow_result["total_time"]

    return results

# Run benchmarks
print("Benchmarking Handoff Patterns")
print("="*60)

benchmark_results = {}
for name, df in datasets.items():
    print(f"\nTesting {name} dataset ({len(df):,} rows)...")
    benchmark_results[name] = benchmark_all_patterns(name, df)

Benchmarking Handoff Patterns

Testing small dataset (10,000 rows)...


Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-05-19 17:47:02,088	INFO parquet_datasource.py:1133 -- Estimated parquet encoding ratio is 1.187.
2026-05-19 17:47:02,096	INFO parquet_datasource.py:1193 -- Estimated parquet reader batch size at 2485514 rows
2026-05-19 17:47:02,124	INFO logging.py:416 -- Registered dataset logger for dataset dataset_10_0
2026-05-19 17:47:02,141	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_10_0. Full logs are in /tmp/ray/session_2026-05-19_17-45-25_832868_4047/logs/ray-data
2026-05-19 17:47:02,146	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_10_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AggregateNumRows[AggregateNumRows]
2026-05-19 17:47:02,246	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_10_0 =======
2026-05-19 17:47:02,250	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:47:02,253	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:


Testing medium dataset (100,000 rows)...


Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-05-19 17:47:11,351	INFO parquet_datasource.py:1133 -- Estimated parquet encoding ratio is 1.216.
2026-05-19 17:47:11,357	INFO parquet_datasource.py:1193 -- Estimated parquet reader batch size at 2485514 rows
2026-05-19 17:47:11,375	INFO logging.py:416 -- Registered dataset logger for dataset dataset_15_0
2026-05-19 17:47:11,383	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_15_0. Full logs are in /tmp/ray/session_2026-05-19_17-45-25_832868_4047/logs/ray-data
2026-05-19 17:47:11,384	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_15_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AggregateNumRows[AggregateNumRows]
2026-05-19 17:47:11,425	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_15_0 =======
2026-05-19 17:47:11,429	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:47:11,432	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:


Testing large dataset (1,000,000 rows)...


Parquet dataset sampling:   0%|          | 0.00/2.00 [00:00<?, ? file/s]

2026-05-19 17:48:08,846	INFO parquet_datasource.py:1133 -- Estimated parquet encoding ratio is 1.219.
2026-05-19 17:48:08,848	INFO parquet_datasource.py:1193 -- Estimated parquet reader batch size at 2485514 rows
2026-05-19 17:48:08,863	INFO logging.py:416 -- Registered dataset logger for dataset dataset_20_0
2026-05-19 17:48:08,871	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_20_0. Full logs are in /tmp/ray/session_2026-05-19_17-45-25_832868_4047/logs/ray-data
2026-05-19 17:48:08,872	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_20_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AggregateNumRows[AggregateNumRows]
2026-05-19 17:48:08,910	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_20_0 =======
2026-05-19 17:48:08,912	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 17:48:08,913	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 17:

In [ ]:
# Display results
print("\nHandoff Pattern Benchmark Results (seconds)")
print("="*60)

results_df = pd.DataFrame(benchmark_results).T
results_df = results_df.replace(float('inf'), 'N/A')
print(results_df.to_string())

print("\n" + "="*60)
print("RECOMMENDATIONS:")
print("  Small (< 100K rows):   Pandas transfer is fastest")
print("  Medium (100K-1M rows): Arrow or Parquet")
print("  Large (> 1M rows):     Parquet (most robust)")


Handoff Pattern Benchmark Results (seconds)
         Parquet    Pandas     Arrow
small   2.504959  1.119534  1.143864
medium  1.758403  3.855915  2.474413
large   4.351253       N/A  4.923108

RECOMMENDATIONS:
  Small (< 100K rows):   Pandas transfer is fastest
  Medium (100K-1M rows): Arrow or Parquet
  Large (> 1M rows):     Parquet (most robust)


---

## 5. Decision Guide

In [ ]:
decision_guide = pd.DataFrame({
    "Pattern": ["Parquet Files", "Pandas In-Memory", "Arrow Direct"],
    "Data Size": ["Any", "< 10 GB", "< 50 GB"],
    "Latency": ["Higher", "Lowest", "Low"],
    "Memory Use": ["Low", "High", "Medium"],
    "Complexity": ["Simple", "Simple", "Medium"],
    "Best For": [
        "Production pipelines",
        "Interactive development",
        "Performance-critical paths"
    ]
})

print("Handoff Pattern Decision Guide")
print("="*80)
print(decision_guide.to_string(index=False))

Handoff Pattern Decision Guide
         Pattern Data Size Latency Memory Use Complexity                   Best For
   Parquet Files       Any  Higher        Low     Simple       Production pipelines
Pandas In-Memory   < 10 GB  Lowest       High     Simple    Interactive development
    Arrow Direct   < 50 GB     Low     Medium     Medium Performance-critical paths


In [ ]:
flowchart = """
CHOOSING A HANDOFF PATTERN
==========================

Data Size?
    │
    ├── < 1 GB and fits in memory?
    │        │
    │        ├── Need lowest latency? ──> PANDAS
    │        │
    │        └── Otherwise ──> ARROW
    │
    ├── 1-50 GB?
    │        │
    │        ├── Need to persist? ──> PARQUET
    │        │
    │        └── One-time transfer? ──> ARROW
    │
    └── > 50 GB?
             │
             └── Always use PARQUET
                 (memory-safe, persistent)


Additional Considerations:
  • Need to resume on failure? ──> PARQUET
  • Need partitioning? ──> PARQUET
  • Shared filesystem available? ──> PARQUET
  • Single-machine, ephemeral? ──> PANDAS or ARROW
"""
print(flowchart)


CHOOSING A HANDOFF PATTERN

Data Size?
    │
    ├── < 1 GB and fits in memory?
    │        │
    │        ├── Need lowest latency? ──> PANDAS
    │        │
    │        └── Otherwise ──> ARROW
    │
    ├── 1-50 GB?
    │        │
    │        ├── Need to persist? ──> PARQUET
    │        │
    │        └── One-time transfer? ──> ARROW
    │
    └── > 50 GB?
             │
             └── Always use PARQUET
                 (memory-safe, persistent)


Additional Considerations:
  • Need to resume on failure? ──> PARQUET
  • Need partitioning? ──> PARQUET
  • Shared filesystem available? ──> PARQUET
  • Single-machine, ephemeral? ──> PANDAS or ARROW



---

## 6. Exercise: Choose the Right Pattern

In [ ]:
exercises = [
    {
        "scenario": "500 MB DataFrame, need to explore interactively",
        "answer": "Pandas - fits in memory, lowest latency for exploration"
    },
    {
        "scenario": "50 GB ETL output, need to train ML model tomorrow",
        "answer": "Parquet - persistent storage, can restart if needed"
    },
    {
        "scenario": "10 GB feature data, training 100 models in parallel",
        "answer": "Parquet - multiple readers can access same files"
    },
    {
        "scenario": "2 GB lookup table, need fast repeated access",
        "answer": "Arrow in object store - ray.put() for sharing"
    },
    {
        "scenario": "500 GB log data, partitioned by date",
        "answer": "Partitioned Parquet - efficient date-based queries"
    },
]

print("Exercise: Choose the Right Handoff Pattern")
print("="*60)
for i, ex in enumerate(exercises, 1):
    print(f"\n{i}. {ex['scenario']}")
    print(f"   → {ex['answer']}")

Exercise: Choose the Right Handoff Pattern

1. 500 MB DataFrame, need to explore interactively
   → Pandas - fits in memory, lowest latency for exploration

2. 50 GB ETL output, need to train ML model tomorrow
   → Parquet - persistent storage, can restart if needed

3. 10 GB feature data, training 100 models in parallel
   → Parquet - multiple readers can access same files

4. 2 GB lookup table, need fast repeated access
   → Arrow in object store - ray.put() for sharing

5. 500 GB log data, partitioned by date
   → Partitioned Parquet - efficient date-based queries


---

## Summary

### Pattern Comparison

| Pattern | Pros | Cons | Use When |
|---------|------|------|----------|
| **Parquet** | Persistent, compressed, partitionable | Disk I/O | Production, large data |
| **Pandas** | Simple, fast for small data | Memory limited | Development, < 10 GB |
| **Arrow** | Efficient columnar format | Setup complexity | Performance critical |

### Best Practices

1. **Default to Parquet** for production pipelines
2. **Use Pandas** for interactive development with small data
3. **Partition by query columns** (date, category) for efficient reads
4. **Test with production-size data** before deploying

### Next: Module 5 - Ray on SLURM

See `05_ray_on_slurm.md` for deploying Ray on HPC.

In [ ]:
# Cleanup
spark.stop()
ray.shutdown()
shutil.rmtree(TEMP_DIR, ignore_errors=True)
print("Cleanup complete.")

Cleanup complete.
